# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing. Instead of running the pruning on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the pruning on more powerful instances.

## What is Pruning?

Pruning is a technique that removes unnecessary weights from a neural network, effectively making the model more sparse. Research has shown that many neural networks are overparameterized, and a significant percentage of weights can be removed without substantial impact on accuracy.

### Types of Pruning

#### Unstructured Pruning
- **Description**: Removes individual weights based on importance criteria (typically magnitude)
- **Advantages**: Higher theoretical compression rates, more fine-grained control
- **Disadvantages**: Requires specialized hardware/software for speed benefits
- **Example**: Setting the smallest 30% of weights to zero based on their absolute values

#### Structured Pruning
- **Description**: Removes entire structures like neurons, channels, or attention heads
- **Advantages**: Immediate speed benefits on standard hardware, actual size reduction
- **Disadvantages**: Generally higher accuracy impact than unstructured pruning
- **Example**: Removing entire neurons or attention heads based on their importance

In this notebook, we'll focus on structured pruning to achieve actual size reduction and inference speedup.

### Benefits of Pruning:
- **Reduced Model Size**: Fewer parameters means smaller models
- **Faster Inference**: Fewer computations lead to faster inference
- **Lower Memory Requirements**: Sparse models require less memory
- **Reduced Overfitting**: Removing redundant weights can improve generalization

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform pruning on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

First, we'll import the necessary libraries for our pruning tasks.

In [ ]:
# Import required libraries
import os
import sys
import time
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import specific modules for this notebook
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output

## 2. Load Workshop Settings

Next, we'll load the workshop settings that were configured in the setup notebook. These settings include the S3 bucket, AWS region, SageMaker role ARN, and instance type for optimization jobs.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

# Initialize S3 client
s3_client = boto3.client('s3')

## 3. Load Model Information

Now we'll load information about the models we want to prune. This information includes the model name, task type, and S3 URI where the model is stored. If the model information file doesn't exist, we'll create a default one with a sentiment analysis model.

In [ ]:
# Load model information from previous notebooks
try:
    with open('model_info.json', 'r') as f:
        model_info_dict = json.load(f)
    print(f"Loaded model information for {len(model_info_dict)} models")
except FileNotFoundError:
    print("model_info.json not found. Creating default model info.")
    model_info_dict = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }
    
    # Save model info to file
    with open('model_info.json', 'w') as f:
        json.dump(model_info_dict, f, indent=2)
    print("Created default model info with sentiment analysis model")

# Display model info
for model_key, info in model_info_dict.items():
    print(f"\nModel: {model_key}")
    print(f"  Name: {info['model_name']}")
    print(f"  Task: {info['task']}")
    print(f"  S3 URI: {info.get('s3_uri', 'Not available')}")

## 4. Model Selection for Pruning

Before configuring our pruning jobs, we need to carefully select which models are suitable for pruning. Based on extensive research and experimentation, we've found that not all model architectures respond well to pruning techniques.

### Models Not Suitable for Pruning

**Named Entity Recognition (NER) Models**: Token classification models like BERT-based NER are highly sensitive to pruning due to:

1. **Token-level Classification Sensitivity**: NER models make token-by-token predictions that rely heavily on contextual relationships between tokens
2. **Attention Mechanism Importance**: The attention mechanisms in transformer models are critical for capturing token relationships, and pruning disrupts these mechanisms
3. **Entity Type Sensitivity**: Different entity types (Person, Organization, Location) show varying levels of sensitivity to pruning
4. **Boundary Detection Issues**: Even with minimal pruning (3%), entity boundary detection is significantly affected

For these models, we recommend alternative optimization approaches like quantization or knowledge distillation instead.

### Filtering Models for Pruning

We'll filter our model list to exclude NER/token-classification models before proceeding with pruning:

In [ ]:
# Load workshop configuration
with open('workshop_config.json', 'r') as f:
    workshop_config = json.load(f)

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = workshop_config['role']
region = workshop_config['region']
bucket = workshop_config['s3_bucket']
prefix = workshop_config['s3_prefix']

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")


In [ ]:
# Filter out models that are not suitable for pruning
prunable_models = {}
excluded_models = {}

for model_key, info in model_info_dict.items():
    if info['task'] == 'token-classification':
        excluded_models[model_key] = info
        print(f"Excluding {model_key} ({info['model_name']}) from pruning as token-classification models are not suitable for pruning")
    else:
        prunable_models[model_key] = info
        print(f"Including {model_key} ({info['model_name']}) for pruning")

print(f"\nSelected {len(prunable_models)} models for pruning out of {len(model_info_dict)} total models")

# Check if we have any models to prune
if len(prunable_models) == 0:
    print("\n⚠️ No suitable models found for pruning. Please add models with supported tasks.")
    print("Supported tasks include: text-classification, question-answering, etc.")
    print("Token-classification (NER) models are not recommended for pruning.")

## 5. Configure Pruning Jobs

Now that we've filtered our models to include only those suitable for pruning, we'll configure SageMaker Processing jobs to perform the pruning. The process involves:

1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each suitable model**:
   - Save and upload model information to S3
   - Define inputs (pruning script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using structured pruning with a pruning amount of 0.3 (30% of weights will be removed). This method removes entire neurons/filters, which actually reduces the model size unlike unstructured pruning.

### Parallel Processing
To speed up the process, we'll launch all jobs in parallel rather than waiting for each job to complete before starting the next one.

In [ ]:
# Launch pruning jobs for all suitable models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing pruning jobs for suitable models...")

for model_key in prunable_models.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: prunable_models[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='pruned-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-method', 'structured',
            '--pruning-amount', '0.3'
        ],
        'output_path': output_path
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching pruning jobs in parallel...")

# Check if we have any models to prune
if len(job_configs) == 0:
    print("No suitable models to prune. Skipping job launch.")
else:
    for model_key, config in job_configs.items():
        try:
            # Create a unique job name with timestamp to avoid conflicts
            timestamp = int(time.time())
            job_name = f"pruning-{model_key}-{timestamp}"
            
            # Run the processing job with the unique name
            processor.run(
                code='pruning_script.py',
                source_dir='pruning_scripts',
                inputs=config['inputs'],
                outputs=config['outputs'],
                arguments=config['arguments'],
                wait=False,
                job_name=job_name
            )
            
            # Store the job name for tracking
            job_names.append(job_name)
            print(f"Launched job for {model_key}: {job_name}")
        except Exception as e:
            print(f"Error launching job for {model_key}: {e}")

    print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

## 5. Monitor Job Status

After launching the pruning jobs, we need to monitor their status to know when they're complete. We'll create a function to check the status of all jobs and display it in a user-friendly way.

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion if there are any jobs running
if len(job_names) > 0:
    print("Waiting for all jobs to complete...")
    while True:
        all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
        
        # Clear previous output
        clear_output(wait=True)
        
        # Print current status
        print("Current job statuses:")
        for job_name, status in job_statuses.items():
            print(f"Job {job_name}: {status}")
        
        if all_complete:
            print("All jobs completed!")
            break
        
        print("Waiting for jobs to complete... Will check again in 60 seconds.")
        time.sleep(60)  # Check every minute

    print("\nAll jobs have completed or failed.")
else:
    print("No pruning jobs were launched. Skipping job monitoring.")

## 6. Analyze Pruned Models

Now that the pruning jobs are complete, we'll analyze the pruned models. We'll use S3 metadata to calculate the actual size reduction without needing to download the models.

In [ ]:
# Create an analysis that uses S3 metadata to calculate actual size reduction
import os
import json
import pandas as pd
import boto3

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

# List to store data for DataFrame
comparison_data = []

for model_key, job_info in job_configs.items():
    print(f"\nAnalyzing model: {model_key}")
    
    # Get model info
    model_info = model_info_dict[model_key]
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Get the S3 URI for the original model
    original_s3_uri = model_info.get("s3_uri")
    if original_s3_uri:
        # Parse the S3 URI to get bucket and prefix
        original_uri_parts = original_s3_uri.replace("s3://", "").split("/")
        original_bucket = original_uri_parts[0]
        original_prefix = "/".join(original_uri_parts[1:])
        
        # Get the S3 URI for the pruned model
        pruned_s3_uri = job_info["output_path"]
        pruned_uri_parts = pruned_s3_uri.replace("s3://", "").split("/")
        pruned_bucket = pruned_uri_parts[0]
        pruned_prefix = "/".join(pruned_uri_parts[1:])
        
        # Get the total size of the original model
        print(f"Calculating size of original model in S3...")
        original_size = get_total_size(original_bucket, original_prefix)
        original_size_mb = original_size / (1024 * 1024)  # Convert to MB
        
        # Get the total size of the pruned model
        print(f"Checking for pruned model in S3 at {pruned_s3_uri}...")
        pruned_size = get_total_size(pruned_bucket, pruned_prefix)
        
        # Handle case where pruned model wasn't created successfully
        if pruned_size == 0:
            print(f"WARNING: No pruned model found at {pruned_s3_uri}")
            print(f"The pruning job for {model_key} may have failed silently.")
            print(f"Using original model size for comparison (no reduction)")
            pruned_size = original_size
            pruned_size_mb = original_size_mb
            size_reduction = 0
        else:
            pruned_size_mb = pruned_size / (1024 * 1024)  # Convert to MB
            # Calculate size reduction
            if original_size > 0:
                size_reduction = (original_size - pruned_size) / original_size * 100
            else:
                size_reduction = 0
        print(f"Calculating size of pruned model in S3...")
        pruned_size = get_total_size(pruned_bucket, pruned_prefix)
        pruned_size_mb = pruned_size / (1024 * 1024)  # Convert to MB
        
        # Calculate size reduction
        if original_size > 0:
            size_reduction = (original_size - pruned_size) / original_size * 100
        else:
            size_reduction = 0
            
        print(f"Model: {model_name}")
        print(f"Task: {task}")
        print(f"Original size: {original_size_mb:.2f} MB")
        print(f"Pruned size: {pruned_size_mb:.2f} MB")
        print(f"Size reduction: {size_reduction:.2f}%")
        
        # For parameter reduction and sparsity, we'll use the pruning amount as an estimate
        # since we can't calculate these without loading the models
        pruning_amount = 0.3  # 30% pruning
        estimated_param_reduction = pruning_amount * 100  # Convert to percentage
        estimated_sparsity = pruning_amount * 100  # Same as parameter reduction for structured pruning
        
        print(f"Pruning amount: {pruning_amount * 100:.1f}%")
        print(f"Estimated parameter reduction: {estimated_param_reduction:.1f}%")
        print(f"Estimated sparsity: {estimated_sparsity:.1f}%")
        
        # Add data for this model to the comparison data list
        comparison_data.append({
            'Model': model_name,
            'Original Size (MB)': round(original_size_mb, 2),
            'Pruned Size (MB)': round(pruned_size_mb, 2),
            'Size Reduction (%)': round(size_reduction, 2),
            'Est. Parameter Reduction (%)': round(estimated_param_reduction, 1),
            'Est. Sparsity (%)': round(estimated_sparsity, 1)
        })
    else:
        print(f"No S3 URI found for model {model_key}, skipping size analysis")

# Create and display DataFrame
if len(comparison_data) > 0:
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df)
else:
    print("No pruned models to analyze. This could be because:")
    print("1. No suitable models were found for pruning (e.g., only NER models were available)")
    print("2. The pruning jobs failed to complete successfully")
    print("3. The pruned models were not saved to S3 correctly")

## 7. Visualize Results

Now we'll visualize the results of our pruning experiments to better understand the impact of pruning on model size. If we have pruned models to analyze, we'll create charts comparing their size and performance.

Note: If no models were pruned (e.g., because only NER models were available), we'll skip the visualization step.

In [ ]:
# Check if we have any data to visualize
if 'comparison_df' in locals() and len(comparison_df) > 0:
    # Set the style
    sns.set(style="whitegrid")

    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot size reduction
    sns.barplot(x='Model', y='Size Reduction (%)', data=comparison_df, ax=ax1, palette='viridis')
    ax1.set_title('Model Size Reduction (%)', fontsize=14)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.set_ylabel('Size Reduction (%)', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)

    # Plot original vs pruned size
    size_data = comparison_df.melt(id_vars=['Model'], 
                                  value_vars=['Original Size (MB)', 'Pruned Size (MB)'],
                                  var_name='Size Type', value_name='Size (MB)')
    sns.barplot(x='Model', y='Size (MB)', hue='Size Type', data=size_data, ax=ax2, palette='viridis')
    ax2.set_title('Model Size Comparison', fontsize=14)
    ax2.set_xlabel('Model', fontsize=12)
    ax2.set_ylabel('Size (MB)', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No pruned models available for visualization.")
    print("This is expected if you only have NER/token-classification models in your model_info.json file.")
    print("To see visualization results, add models with supported tasks like text-classification.")

## 8. Special Considerations for NER Models

Through extensive experimentation, we've discovered that pruning is not suitable for Named Entity Recognition (NER) models based on transformer architectures like BERT. Our findings show that even minimal pruning significantly impacts entity recognition capabilities.

### NER Model Pruning Results

We tested three different pruning configurations on a BERT-based NER model:

1. **Heavy Pruning (30%)**: Complete failure - no entities detected
2. **Light Pruning (10%)**: Partial functionality - only location entities retained (33% of original entities)
3. **Minimal Pruning (3%)**: Maintained functionality but with errors - all entity types retained but with boundary issues

### Why Pruning Fails for NER Models

1. **Token-level Classification Sensitivity**: NER models make token-by-token predictions that are highly sensitive to contextual relationships between tokens
2. **Attention Mechanism Importance**: The attention mechanisms in transformer models are critical for capturing token relationships, and pruning disrupts these mechanisms
3. **Entity Type Sensitivity**: Different entity types (Person, Organization, Location) show varying levels of sensitivity to pruning
4. **Boundary Detection Issues**: Even with minimal pruning, entity boundary detection is significantly affected

### Recommendations for NER Models

- **Avoid Pruning**: Do not apply pruning to NER models
- **Alternative Approaches**: Consider quantization or knowledge distillation instead
- **Smaller Base Models**: If size reduction is necessary, start with a smaller pre-trained model rather than pruning a larger one

## 9. Conclusion

In this notebook, we've applied pruning to our models using SageMaker Processing jobs and learned about its effectiveness for different model types. We've seen that:

1. **Structured pruning** can reduce model size and improve inference time for certain model types
2. **Model architecture matters**: Not all models are suitable for pruning, particularly NER models
3. **Distributed processing** allows us to efficiently test multiple models in parallel
4. **Model analysis** helps us understand the impact of pruning on various metrics

### Key Takeaways

- Pruning effectiveness varies significantly by model architecture and task
- Structured pruning removes entire neurons/filters, resulting in actual size reduction
- The pruning amount should be adjusted based on your specific model and requirements
- Always test pruned models thoroughly to ensure they maintain acceptable performance

### Next Steps

In the next notebook, we'll explore knowledge distillation, another powerful technique for model optimization that involves training a smaller "student" model to mimic the behavior of a larger "teacher" model. This approach may be more suitable for models like NER that don't respond well to pruning.